# 03 — Open questions (the three things you flagged, plus file status)
**Built 2026-09-01.** Your overnight message flagged: (1) the isodepth/pseudotime **transfer**
results, (2) the **uninduced sample**, (3) **spatial QC** not working as you thought.
Each gets a section with everything the artifacts + repo say, ending in a decision.

In [1]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt
for DATA in ["data", "review/data", "../verify/data"]:
    if os.path.isdir(DATA): break
def J(name): return json.load(open(os.path.join(DATA, name)))
print("reading from:", os.path.abspath(DATA))

reading from: /Users/nickhilgert/.claude-science/orgs/9788b220-3ce1-4fba-9010-4e86764515bf/workspaces/d7a6da7b-f450-4f68-b0c4-d439fd5c6a0d/review/data


## Q1 — The three transfer-ρ values do not reconcile (and shouldn't be expected to)
The verify README (Known-open section) flags: **0.389 / 0.591 / 0.688** — "different arms,
coordinate vintages, and cell subsets. Resolve before putting a transfer number in the manuscript."

Verbatim constructions (from the README + verify notebooks):

| value | quantity | direction | subset / vintage |
|---|---|---|---|
| **0.389** | transferred isodepth vs native isodepth (stored column) | mm→hs | vanilla SAMap object, like-for-like |
| **0.591** | transferred vs native isodepth, full object, per-timepoint pooled | hs→mm | full transfer object (see live per-group below) |
| **0.688** | mean of 12 isolated per-timepoint fits | hs→mm | single-timepoint fits (`transfer_results.csv`) |
| 0.792 | human isodepth → scored vs mouse **pseudotime** (cross-scalar) | hs→mm | `coordinate_transfer.json` — a *different quantity* |
| 0.655 | mouse dpt → scored vs human (cross-scalar, reverse) | mm→hs | same file, same caveat |

These differ in **direction**, **target scalar**, **cell subset**, and **coordinate vintage** —
numeric disagreement is expected, not alarming. The error would be quoting any one of them
as "the" transfer number.

In [2]:
ipt = J("isodepth_pt_correlation.json")
d = ipt["mouse_native_iso_vs_transferred_iso"]
print(f"0.591 construction, live: spearman {d['spearman']:.3f} (n={d['n']:,}) — per timepoint:")
for g, gv in d["per_group"].items():
    try: print(f"   {g:14s} {gv['spearman']:+.3f} (n={gv['n']:,})")
    except Exception: print(f"   {g:14s} {gv}")
tr = pd.read_csv(os.path.join(DATA, "transfer_results.csv"))
print("\n0.688 reproduction attempt — means over the current transfer_results.csv (12 fits):")
for col in ["rho_all_hm","rho_all_mh","rho_dec_hm","rho_within_hm","rho_within_mh"]:
    print(f"   mean {col:14s}: all12 {tr[col].mean():.3f}   excl-uninduced {tr[tr.stage!='uninduced'][col].mean():.3f}")
print("-> NONE equals 0.688 (closest: rho_within_mh all12, 0.659).")
print("   The 0.688 value cannot be regenerated from the CURRENT per-timepoint table —")
print("   it must come from an earlier vintage of the fits. This strengthens the case")
print("   for one clean recompute rather than reconciling stale numbers.")

0.591 construction, live: spearman 0.591 (n=43,183) — per timepoint:
   uninduced      +0.711 (n=9,334)
   1_dpi_early    +0.583 (n=2,688)
   1_dpi_late     +0.596 (n=10,787)
   2_dpi          +0.702 (n=3,571)
   3_dpi          +0.793 (n=10,664)
   4_dpi_bleeding +0.207 (n=6,139)

0.688 reproduction attempt — means over the current transfer_results.csv (12 fits):
   mean rho_all_hm    : all12 0.354   excl-uninduced 0.404
   mean rho_all_mh    : all12 0.422   excl-uninduced 0.503
   mean rho_dec_hm    : all12 0.224   excl-uninduced 0.246
   mean rho_within_hm : all12 0.887   excl-uninduced 0.887
   mean rho_within_mh : all12 0.659   excl-uninduced 0.787
-> NONE equals 0.688 (closest: rho_within_mh all12, 0.659).
   The 0.688 value cannot be regenerated from the CURRENT per-timepoint table —
   it must come from an earlier vintage of the fits. This strengthens the case
   for one clean recompute rather than reconciling stale numbers.


> ### DECISION — Q1 transfer reconciliation
> - [ ] **Approve plan** — recompute ONE like-for-like quantity (transferred iso vs native iso, both directions, same cell subset, current coordinate vintage, per timepoint) as a config-driven verify addition; quote only the per-timepoint table in the manuscript, never a pooled scalar
> - [ ] **Quote per-timepoint table only** — skip the recompute; drop 0.389/0.591/0.688 entirely
> - [ ] **Hold** — transfer claims stay out of the manuscript for now
>
> _Notes:_

## Q2 — The uninduced sample: what it can and cannot support
Everything the artifacts say about uninduced, in one place. The pattern: uninduced behaves
**consistently as tissue** (isodepth fits, conservation enrichment) but is **unusable for
transfer/decidual claims** (9 decidual cells; transfer ρ construction-dependent from 0.008 to 0.711).

In [3]:
spec = J("specificity_control.json")
print("1) design caveat  :", spec["design"]["caveat"])
print(f"2) conservation   : uninduced fold {spec['verdict']['uninduced_fold']} vs induced {spec['verdict']['induced_fold_mean']} — present, comparable strength")
print("3) isodepth       : uninduced fit exists (4,155 cells) but NOT seeded (verify README/nb01);")
print("                    excluded from seeding + induced-only embeddings per standing rule")
tr = pd.read_csv(os.path.join(DATA, "transfer_results.csv"))
u = tr[tr.stage=="uninduced"][["phase","rho_all_hm","rho_all_mh","n_dec_mh","p_hm"]]
print("4) transfer (single-timepoint fits):"); print(u.round(3).to_string(index=False))
ipt = J("isodepth_pt_correlation.json")
try:
    uv = ipt["mouse_native_iso_vs_transferred_iso"]["per_group"]["uninduced"]
    print(f"   vs full-object construction: uninduced {uv['spearman']:+.3f} (n={uv['n']:,}) — construction-dependent!")
except KeyError as e:
    print("   full-object per-group key:", list(ipt["mouse_native_iso_vs_transferred_iso"]["per_group"]))
bd = J("boundary_depth_v2_report.json")
for k in ("uninduced_mck8a","uninduced_mck8b"):
    b = bd["blobs"][k]
    print(f"5) {k}: {b['n_cells']:,} cells, {b['n_blobs']} blob(s), {b['n_noise']:,} DBSCAN noise")
print("6) decidual cells in uninduced: 9 (transfer_results.csv, n_dec) — no decidual claims possible")

1) design caveat  : uninduced = 2 sections from 1 animal; pseudo-replicates
2) conservation   : uninduced fold 2.581 vs induced 2.377 — present, comparable strength
3) isodepth       : uninduced fit exists (4,155 cells) but NOT seeded (verify README/nb01);
                    excluded from seeding + induced-only embeddings per standing rule
4) transfer (single-timepoint fits):
    phase  rho_all_hm  rho_all_mh  n_dec_mh  p_hm
secretory       0.190       0.034         9 0.050
menstrual       0.008      -0.006         9 0.965
   vs full-object construction: uninduced +0.711 (n=9,334) — construction-dependent!
5) uninduced_mck8a: 2,142 cells, 2 blob(s), 82 DBSCAN noise
5) uninduced_mck8b: 1,971 cells, 2 blob(s), 73 DBSCAN noise
6) decidual cells in uninduced: 9 (transfer_results.csv, n_dec) — no decidual claims possible


> ### DECISION — Q2 uninduced
> - [ ] **Approve framing** — uninduced = control-only: supports the specificity control and tissue-generic conservation; excluded from ALL transfer/decidual claims; pseudo-replicate caveat (2 sections, 1 animal) attached wherever quoted
> - [ ] **Strengthen** — plan more uninduced tissue (real replication) before the paper
> - [ ] **Drop** — remove uninduced from the narrative entirely (weakens the specificity control)
>
> _Notes:_

## Q3 — Spatial QC: your recollection vs what's implemented
**Your observation was correct** — cells you expected filtered survive into downstream objects.
What the repo actually implements:

- **Stage-01 QC is expression-only**: count floor, MAD outliers, mito ceiling, doublets
  (`workflow/scripts/qc.py`). No spatial criteria at all.
- **Ingest/concat keep every cell**, placed or not, with *nullable* spatial coordinates
  (`x_um_dbscan/y_um_dbscan` are null for unplaced/off-blob cells).
- **DBSCAN placement exists** (hence the `_dbscan` column suffix + `n_noise` counts in the
  boundary report) — but it only *annotates*; nothing global ever drops noise cells.
- **Each downstream arm opts in ad hoc**: the spatial-GP script drops non-finite coords and
  upper-tail pseudotime outliers; the SAMap/expression arms filter nothing (correct for
  expression-only analyses, but it means unplaced cells ride along everywhere).

So "spatial QC" = per-arm filtering, not a shared stage. Live census of the actual objects:

In [4]:
qa = J("h5ad_qc_audit2.json")
rows = []
for p, rec in qa.items():
    if "error" in rec or "spatialish" not in rec: continue
    nn = rec["spatialish"].get("x_um_dbscan", {}).get("n_null")
    if nn is None: continue
    rows.append({"object": p.split("/")[-1], "n_obs": rec["n_obs"], "n_null_coords": nn,
                 "frac_unplaced": round(nn/rec["n_obs"], 3)})
df = pd.DataFrame(rows).sort_values("frac_unplaced", ascending=False)
print(df.to_string(index=False))
print("\nDeliverables carry 30-42% coordinate-less cells; SAMap inputs up to ~41%.")
print("Also note: 08_28 vs 08_05 vintages differ in null counts (placement was updated).")

                                         object  n_obs  n_null_coords  frac_unplaced
2026_08_05_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad  43197          18668          0.432
2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad  43197          18317          0.424
                          mm_fib_dec_2_dpi.h5ad   3574           1447          0.405
                          mm_fib_dec_3_dpi.h5ad  10673           3874          0.363
2026_08_05_Cevrim_Human_Only_Fib_Dec_Cells.h5ad  47533          16237          0.342
2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad  47533          14283          0.300
                      hs_fib_dec_secretory.h5ad  23441           4689          0.200

Deliverables carry 30-42% coordinate-less cells; SAMap inputs up to ~41%.
Also note: 08_28 vs 08_05 vintages differ in null counts (placement was updated).


> ### DECISION — Q3 spatial QC
> - [ ] **Approve new stage** — add config-driven `01d_spatial_qc.smk`: flags (`spatial_qc_pass`: placed + on-blob + optional distance-to-blob threshold) written into the objects + a per-sample report; spatial arms consume the flag; expression arms keep all cells but the flag travels with them (one-morning job to implement, needs your thresholds)
> - [ ] **Document only** — keep per-arm filtering as-is, add a methods paragraph stating expression analyses include unplaced cells
> - [ ] **Stricter** — drop unplaced cells at ingest (changes every downstream object; big blast radius)
>
> _Notes:_

## Q4 — File status roundup
| file | status | action |
|---|---|---|
| `cutoff_dependence.json` | **missing everywhere** (repo, revisions tree, netscratch — exhaustive overnight search) | decision in notebook 01 §10: re-run / quote unverified / drop |
| `rrho_permutation_null.json` | **missing everywhere** (same search) | same decision |
| `prep_samap_input.py.raw_counts.bak` | **safe** — moved to attic by cleanup change 1; caused the single verify mismatch | one-line path fix in verify nb05 (or restore the file) |
| April-dated SAMap pickles (`samap_integrate/runs/`) | unreadable (`KeyError: '_adata_ref'`) — known trap | none; only 2026-08-31 pickles load |
| `2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad` | present; audited above | remember: filename says **XMens**, not Mouse |

No decision needed here beyond notebook 01 §10.